In [1]:
# Run tomes on Nick's isolate genomes
import glob

In [4]:
# from tqdm import tqdm
import pandas as pd
import numpy as np

In [5]:
import os
import subprocess
from IPython.display import display, clear_output
import pickle

In [2]:
gtdb_genomes = glob.glob('/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/validation_datasets/Arthrobacter/ncbi_dataset/data/*/*.fna')
gtdb_genomes[:5]

['/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/validation_datasets/Arthrobacter/ncbi_dataset/data/GCA_004000035.1/GCA_004000035.1_ASM400003v1_genomic.fna',
 '/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/validation_datasets/Arthrobacter/ncbi_dataset/data/GCA_024168675.1/GCA_024168675.1_ASM2416867v1_genomic.fna',
 '/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/validation_datasets/Arthrobacter/ncbi_dataset/data/GCA_030060415.1/GCA_030060415.1_ASM3006041v1_genomic.fna',
 '/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/validation_datasets/Arthrobacter/ncbi_dataset/data/GCA_030818145.1/GCA_030818145.1_ASM3081814v1_genomic.fna',
 '/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/validation_datasets/Arthrobacter/ncbi_dataset/data/GCA_030946705.1/GCA_030946705.1_ASM3094670v1_genomic.fna']

In [18]:
base = '/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/validation_datasets/Arthrobacter/gtdb_genomes/'
for i, infile in enumerate(gtdb_genomes):
    display(i)
    clear_output(wait=True)
    
    outfile_base = os.path.join(base, infile.split('/')[-2])
    command = f'cp {infile} {outfile_base}_genomic.fna'
    result = subprocess.run(command, shell=True, capture_output=True)
    command = f'prodigal -i {infile} -c -a {outfile_base}_proteins.faa -d {outfile_base}_proteins.fna -o {outfile_base}_proteins.gbk -p single'
    result = subprocess.run(command, shell=True, capture_output=True)

117

In [ ]:
    prodigal -i "${base}_filtered.fasta" -c \
             -a "${base}_proteins.faa" \
             -d "${base}_proteins.fna" \
             -o "${base}_prodigal_output.gbk" \
             -p meta

In [6]:
isolate_proteins = glob.glob('/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/validation_datasets/Arthrobacter/genomes/*.faa')

In [7]:
isolate_proteins[:5]

['/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/validation_datasets/Arthrobacter/genomes/8HZ65L_1_ASP410-2_proteins.faa',
 '/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/validation_datasets/Arthrobacter/genomes/8HZ65L_3_24664_proteins.faa',
 '/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/validation_datasets/Arthrobacter/genomes/barcode01_consensus_proteins.faa',
 '/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/validation_datasets/Arthrobacter/genomes/barcode02_consensus_proteins.faa',
 '/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/validation_datasets/Arthrobacter/genomes/barcode03_consensus_proteins.faa']

# Code for retraining Tome

In [6]:
# load temperature data
temp_data = pd.read_csv('/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/metric_tables/20240327_metrics_df_full.tsv.gz', sep='\t', index_col=0)
temp_data.head()

genome2temp = temp_data.set_index('uid')[['temp']]

In [7]:
# load GTDB metadata
gtdb_md = pd.read_csv('/data/mhoffert/genomes/GTDB_r214.1/bac120_metadata.tsv.gz', sep='\t', compression='gzip')

/tmp/ipykernel_497811/991770083.py:2: DtypeWarning: Columns (61,63,65,74,82,83) have mixed types. Specify dtype option on import or set low_memory=False.
  gtdb_md = pd.read_csv('/data/mhoffert/genomes/GTDB_r214.1/bac120_metadata.tsv.gz', sep='\t', compression='gzip')


In [8]:
with open('./../data/predictive_models/supporting_data/even_msa_dataset.pickle', 'rb') as handle:
    even_msa_dataset = pickle.load(handle)
    
# get GTDB accessions for our dataset
msa_keys = dict((i[0].split('|')[1], i[0]) for i in even_msa_dataset)

print(f'There are {len(even_msa_dataset)} keys in this dataset')

There are 1227 keys in this dataset


In [9]:
# make mapping of genome to taxonomy
genome2tax = gtdb_md.set_index('accession').loc[msa_keys.keys(), 'gtdb_taxonomy']
# make sure all records are found
print(len(msa_keys), len(genome2tax))

# split levels into cols
levels = ['domain', 'phylum', 'class', 'order', 'family', 'genus', 'species']
genome2tax = genome2tax.apply(lambda x: pd.Series(index=levels, data=[i.split('__')[-1] for i in x.split(';')]))
# map index to sequences
genome2tax = genome2tax.rename(index=msa_keys)

genome2tax.head()

1227 1227


,domain,phylum,class,order,family,genus,species
accession,,,,,,,
LBMP01000009.1_70|GB_GCA_001184205.1|adk|temp=30.0|bitscore=199.8,Bacteria,Desulfobacterota,Desulfarculia,Desulfarculales,Desulfarculaceae,Desulfocarbo,Desulfocarbo indianensis
AP018180.1_3248|GB_GCA_002368155.1|adk|temp=20.0|bitscore=179.1,Bacteria,Cyanobacteriota,Cyanobacteriia,Cyanobacteriales,Nostocaceae,Aulosira,Aulosira carnea
WESA01000010.1_17|GB_GCA_009768935.1|adk|temp=40.0|bitscore=153.8,Bacteria,Spirochaetota,Spirochaetia,Treponematales,Rectinemataceae,Rectinema,Rectinema subterraneum
WMEX01000003.1_136|GB_GCA_009856365.1|adk|temp=27.0|bitscore=189.4,Bacteria,Pseudomonadota,Gammaproteobacteria,Pseudomonadales,Oleiphilaceae,Halospina,Halospina utahensis_B
CP031518.1_212|GB_GCA_017161305.1|adk|temp=37.0|bitscore=176.2,Bacteria,Spirochaetota,Spirochaetia,Treponematales,Treponemataceae,Treponema_D,Treponema_D ruminis


In [30]:
with open('/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/validation_datasets/tomes/corkrey_genome_ids.txt', 'r') as handle:
    ids = [l.strip() for l in handle.readlines()]
print(len(ids))

# assert len(set(msa_keys.keys()).intersection(set(ids))) == len(set(ids)), 'uh oh!'

230


In [33]:
# copy genomes from even dataset to a folder
base = '/data/mhoffert/genomes/GTDB_r214.1/protein_faa_reps/bacteria/'
out_base = '/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/validation_datasets/tomes/corkrey_genomes/'
for i in ids: #tqdm(ids):
    command = f'gunzip -c {base}{i}_protein.faa.gz > {out_base}{i}_protein.faa'
    result = subprocess.run(command, shell=True, capture_output=True)

#### Notes
make sure to run this:
```
for i in *.faa; do mv "$i" "${i/faa/fasta}"; done
```

In [34]:
# get original data to match format
tomes_training_data = pd.read_csv('~/tools/Tome/tome/data/training_csvs/train.csv')
print(f'Tome original training data, {tomes_training_data.shape}')
tomes_training_data.head()

Tome original training data, (5533, 402)


,Organisms,AA,AC,AD,AE,AF,AG,AH,AI,AK,...,YN,YP,YQ,YR,YS,YT,YV,YW,YY,ogt
0,37_allobaculum_stercoricanis_bacteria,0.005367,0.001105,0.003809,0.003263,0.002990,0.004836,0.001432,0.006074,0.005794,...,0.001727,0.001648,0.001823,0.001773,0.002474,0.002147,0.002305,0.000403,0.001534,37
1,28_shewanella_algae_bacteria,0.010752,0.001212,0.005580,0.007041,0.003622,0.007179,0.001718,0.005544,0.005156,...,0.001011,0.001440,0.002096,0.002270,0.002069,0.001223,0.001472,0.000515,0.000947,28
2,37_bifidobacterium_saguini_bacteria,0.013873,0.001048,0.007266,0.006288,0.003457,0.008843,0.002224,0.005884,0.005065,...,0.000994,0.001298,0.001138,0.001713,0.001673,0.001711,0.002072,0.000453,0.000926,37
3,28_ochrobactrum_rhizosphaerae_bacteria,0.013460,0.000835,0.006199,0.006978,0.004413,0.009027,0.002097,0.007038,0.004678,...,0.000781,0.001143,0.000817,0.001730,0.001412,0.001173,0.001715,0.000378,0.000642,28
4,30_sphingomonas_haloaromaticamans_bacteria,0.022561,0.001093,0.008958,0.008309,0.004629,0.013291,0.002516,0.007588,0.003563,...,0.000569,0.001035,0.000689,0.002068,0.001160,0.000992,0.001508,0.000357,0.000635,30


In [35]:
import sys
sys.path.append('/data/mhoffert/tools/Tome/tome/')
from core.predOGT import get_dimer_frequency

In [36]:
genome2temp.head()

,temp
uid,
GB_GCA_000376885.1,37.0
GB_GCA_000016765.1,30.0
GB_GCA_000242235.1,37.0
GB_GCA_000283575.1,30.0
GB_GCA_001054945.1,37.0


In [109]:
all_dimers = []
base = '/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/validation_datasets/tomes/even_genomes/'
genomes = glob.glob(f'{base}*.fasta')
print(f'Found {len(genomes)} genomes')
for g in tqdm(genomes):
    genome_id = g.split('/')[-1].replace('_protein.fasta', '')s
    dimers = pd.Series(get_dimer_frequency(g, 5))
    dimers.loc['Organisms'] = genome_id
    dimers.loc['ogt'] = genome2temp.loc[genome_id, 'temp']
    all_dimers.append(dimers)
    # dimers[~dimers.index.str.contains('X')]

Found 1227 genomes


100%|████████████████████████████████████████████████████████████████| 1227/1227 [48:27<00:00,  2.37s/it]


In [111]:
even_tome_data = pd.concat(all_dimers, axis=1).T.reindex(columns=tomes_training_data.columns) 

In [113]:
# add a check here
print(even_tome_data.shape)
even_tome_data.head()

(1227, 402)


,Organisms,AA,AC,AD,AE,AF,AG,AH,AI,AK,...,YN,YP,YQ,YR,YS,YT,YV,YW,YY,ogt
0,GB_GCA_001184205.1,0.014968,0.00154,0.005152,0.007366,0.003674,0.010953,0.001946,0.004308,0.004953,...,0.000754,0.0013,0.001265,0.001889,0.001327,0.001086,0.001647,0.000426,0.000905,30.0
1,GB_GCA_002368155.1,0.007789,0.000792,0.004058,0.00543,0.002946,0.005408,0.001337,0.007105,0.004478,...,0.001254,0.001622,0.002203,0.002011,0.002029,0.001545,0.001647,0.000553,0.001153,20.0
2,GB_GCA_002368275.1,0.007534,0.000798,0.004024,0.005312,0.00292,0.00527,0.00129,0.007124,0.004452,...,0.001317,0.001638,0.002263,0.001955,0.002054,0.001539,0.001624,0.000548,0.00115,20.0
3,GB_GCA_009768935.1,0.01165,0.001152,0.004822,0.006242,0.004602,0.008361,0.002112,0.007053,0.005052,...,0.001033,0.001505,0.000923,0.002145,0.00197,0.001464,0.0018,0.000431,0.000984,40.0
4,GB_GCA_009856365.1,0.009993,0.001012,0.005774,0.007039,0.0036,0.008958,0.002053,0.004911,0.00208,...,0.000784,0.001309,0.001206,0.002445,0.001441,0.00129,0.00141,0.000435,0.000712,27.0


In [114]:
even_tome_data.to_csv('/data/mhoffert/tools/Tome/tome/data/even_train2.csv', index=False)

# code to retrain tomes in a loop
for CV

In [37]:
even_tome_data = pd.read_csv('/data/mhoffert/tools/Tome/tome/data/training_csvs/even_train2.csv')
even_tome_data

,Organisms,AA,AC,AD,AE,AF,AG,AH,AI,AK,...,YN,YP,YQ,YR,YS,YT,YV,YW,YY,ogt
0,GB_GCA_001184205.1,0.014968,0.001540,0.005152,0.007366,0.003674,0.010953,0.001946,0.004308,0.004953,...,0.000754,0.001300,0.001265,0.001889,0.001327,0.001086,0.001647,0.000426,0.000905,30.0
1,GB_GCA_002368155.1,0.007789,0.000792,0.004058,0.005430,0.002946,0.005408,0.001337,0.007105,0.004478,...,0.001254,0.001622,0.002203,0.002011,0.002029,0.001545,0.001647,0.000553,0.001153,20.0
2,GB_GCA_002368275.1,0.007534,0.000798,0.004024,0.005312,0.002920,0.005270,0.001290,0.007124,0.004452,...,0.001317,0.001638,0.002263,0.001955,0.002054,0.001539,0.001624,0.000548,0.001150,20.0
3,GB_GCA_009768935.1,0.011650,0.001152,0.004822,0.006242,0.004602,0.008361,0.002112,0.007053,0.005052,...,0.001033,0.001505,0.000923,0.002145,0.001970,0.001464,0.001800,0.000431,0.000984,40.0
4,GB_GCA_009856365.1,0.009993,0.001012,0.005774,0.007039,0.003600,0.008958,0.002053,0.004911,0.002080,...,0.000784,0.001309,0.001206,0.002445,0.001441,0.001290,0.001410,0.000435,0.000712,27.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1222,RS_GCF_904846055.1,0.009948,0.001031,0.005475,0.004795,0.003432,0.006166,0.001908,0.006585,0.005996,...,0.001177,0.001431,0.002079,0.001537,0.001605,0.001082,0.001606,0.000381,0.001038,29.0
1223,RS_GCF_904846075.1,0.008605,0.000931,0.005678,0.005139,0.003245,0.006353,0.001959,0.006407,0.004947,...,0.001280,0.001519,0.001881,0.001645,0.001823,0.001543,0.001865,0.000410,0.001144,15.0
1224,RS_GCF_904846155.1,0.008720,0.000938,0.005622,0.005258,0.003113,0.006233,0.001996,0.006240,0.005068,...,0.001253,0.001543,0.001933,0.001745,0.001860,0.001560,0.001841,0.000426,0.001095,25.0
1225,RS_GCF_904846715.1,0.008409,0.000922,0.005642,0.004939,0.003163,0.006532,0.001984,0.006323,0.004854,...,0.001272,0.001480,0.001954,0.001681,0.001830,0.001457,0.001932,0.000433,0.001120,22.0


In [38]:
even_ids = [i[0] for i in even_msa_dataset]
base = '/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/validation_datasets/tomes/even_genomes/'
out_base = '/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/validation_datasets/tomes/tome_retraining/'

In [ ]:
subset =  genome2tax.loc[[i[0] for i in even_msa_dataset], :].copy()
logfile_text = ''
for level_i, level in enumerate(subset.columns[2:]):
    print(level)
    for taxon in subset.loc[:, level].unique():
        display(level, taxon)
        logfile_text += '-'*50+'\n'
        logfile_text += taxon+'\n'
        logfile_text += '-'*50+'\n'
        train_inds = [i for i in range(len(even_msa_dataset)) if subset.iloc[i, level_i+1] != taxon]
        test_inds = [i for i in range(len(even_msa_dataset)) if subset.iloc[i, level_i+1] == taxon]
                
        # add assert here to ensure no overlap of train and test
        assert set(train_inds).intersection(set(test_inds)) == set(), 'train and test overlap'
        
        # X_train = scaled_even_full[train_inds, :]
        # y_train = even_temps[train_inds]
        # X_test = scaled_even_full[test_inds, :]
        # y_test = even_temps[test_inds]

        even_tome_data.set_index('Organisms', drop=True).loc[[i.split('|')[1] for i in np.array(even_ids)[train_inds]], :].to_csv('/data/mhoffert/tools/Tome/tome/data/train.csv')

        out_str = f'{taxon}_Tome_predictions.tsv'
        print('Running command 1')
        command1 = 'tome predOGT --train'
        result1 = subprocess.run(command1, shell=True, capture_output=True)
        logfile_text += result1.stdout.decode()
        
        with open(os.path.join(out_base, 'retrain_logfile.txt'), 'w') as handle:
            handle.write(logfile_text)
        
        print('Running command 2')
        command2 = f'tome predOGT --indir {base} --out {os.path.join(out_base, out_str)} -p 8 &'
        result2 = subprocess.run(command2, shell=True, capture_output=True)
        logfile_text += result2.stdout.decode()
        
        print('Running command 3')
        command3 = 'rm /data/mhoffert/tools/Tome/tome/data/train.csv'
        result3 = subprocess.run(command3, shell=True, capture_output=True)

        with open(os.path.join(out_base, 'retrain_logfile_cont.txt'), 'w') as handle:
            handle.write(logfile_text)
        
        clear_output(wait=True)

'order'

'Aquificales'

Running command 1
Running command 2


In [25]:
result1.stdout.decode()

'A new model has beed successfully trained.\nModel performance:\n        RMSE: 2.538184842844792\n          r2: 0.9661744020225317\n  Pearson r:PearsonRResult(statistic=0.9832943088763153, pvalue=0.0)\n  Spearman r:SignificanceResult(statistic=0.9709162499279422, pvalue=0.0)\n\nSaving the new model to replace the original one...\nDone!\n\n'

In [ ]:
# make the plot
